# SMS Spam Classifier — Transfer Learning with DistilBERT
Run each cell **top to bottom**. Each cell = one phase.

| Cell | Phase |
|---|---|
| 1 | Install dependencies |
| 2 | Download dataset |
| 3 | Explore data (EDA) |
| 4 | Tokenise + split 70/15/15 |
| 5 | Fine-tune DistilBERT |
| 6 | Evaluate on validation set |
| 7 | Freeze layers + re-tune |
| 8 | Final test (run once) |
| 9 | Run FastAPI server |

## Cell 1 — Install dependencies

In [ ]:
import sys
!{sys.executable} -m pip install -q transformers torch datasets scikit-learn pandas numpy mlflow fastapi uvicorn python-multipart matplotlib seaborn tqdm accelerate
print('All dependencies installed.')

## Cell 2 — Download dataset

In [ ]:
import urllib.request, zipfile, pandas as pd
from pathlib import Path

RAW_DIR = Path('data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = RAW_DIR / 'spam.csv'

if OUTPUT_CSV.exists():
    print(f'Already exists: {OUTPUT_CSV}')
else:
    UCI_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip'
    zip_path = RAW_DIR / 'tmp.zip'
    print('Downloading from UCI...')
    urllib.request.urlretrieve(UCI_URL, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        with z.open('SMSSpamCollection') as f:
            lines = f.read().decode('utf-8', errors='replace').splitlines()
    rows = [line.split('\t', 1) for line in lines if '\t' in line]
    df = pd.DataFrame(rows, columns=['label', 'text'])
    df.to_csv(OUTPUT_CSV, index=False)
    zip_path.unlink()
    print(f'Saved {len(df)} rows to {OUTPUT_CSV}')

df = pd.read_csv(OUTPUT_CSV)
print(df.head())

## Cell 3 — Full EDA: Feature Engineering + 8 Visualizations

Builds 10 features from raw text and produces charts that explain the data to anyone at a glance.
Run sub-cells **3a → 3b → 3c → 3d → 3e → 3f → 3g → 3h → 3i** in order.

In [ ]:

# ── 3a: Load data and engineer 10 text features ───────────────────────────
import re, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sns.set_theme(style='whitegrid', palette='muted')
COLORS = {'ham': '#4C9BE8', 'spam': '#E8614C'}

df = pd.read_csv('data/raw/spam.csv', encoding='latin-1')
if 'v1' in df.columns:
    df = df[['v1','v2']].rename(columns={'v1':'label','v2':'text'})
df = df[['label','text']].dropna()
df['label']    = df['label'].str.strip().str.lower()
df['label_id'] = (df['label'] == 'spam').astype(int)

# Feature engineering
df['char_count']    = df['text'].str.len()
df['word_count']    = df['text'].str.split().str.len()
df['unique_words']  = df['text'].apply(lambda x: len(set(x.lower().split())))
df['upper_count']   = df['text'].apply(lambda x: sum(1 for c in x if c.isupper()))
df['upper_ratio']   = df['upper_count'] / df['char_count'].clip(lower=1)
df['digit_count']   = df['text'].apply(lambda x: sum(c.isdigit() for c in x))
df['exclaim_count'] = df['text'].str.count('!')
df['punct_count']   = df['text'].apply(lambda x: sum(1 for c in x if c in '!?.,;:'))
df['has_url']       = df['text'].str.contains(r'http|www|\.com|\.net|\.org', case=False, regex=True).astype(int)
df['has_currency']  = df['text'].str.contains(r'[£$€¥]|free|win|prize|cash|award', case=False, regex=True).astype(int)

print(f'Rows: {len(df)}  |  Ham: {(df.label=="ham").sum()}  |  Spam: {(df.label=="spam").sum()}')
print('\nMean feature values by class:')
display(df.groupby('label')[['char_count','word_count','upper_ratio',
                              'exclaim_count','has_url','has_currency']].mean().round(3))


In [ ]:

# ── 3b: Chart 1 — Class balance + message length distribution ─────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Bar: class counts
counts = df['label'].value_counts()
bars = axes[0].bar(counts.index, counts.values,
                   color=[COLORS[l] for l in counts.index], edgecolor='white', width=0.5)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
                 f'{val}\n({100*val/len(df):.1f}%)', ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of messages')
axes[0].set_ylim(0, counts.max() * 1.25)

# Pie: class split
axes[1].pie(counts.values, labels=['Ham (legit)', 'Spam'],
            colors=[COLORS['ham'], COLORS['spam']],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Ham vs Spam Split', fontsize=13, fontweight='bold')

# KDE: message length
for label, grp in df.groupby('label'):
    grp['char_count'].plot.kde(ax=axes[2], label=f'{label.capitalize()} (median={grp["char_count"].median():.0f})',
                                color=COLORS[label], linewidth=2.5)
    axes[2].axvline(grp['char_count'].median(), color=COLORS[label], linestyle='--', alpha=0.5)
axes[2].set_xlabel('Message length (chars)')
axes[2].set_xlim(0, 200)
axes[2].set_title('Message Length Distribution', fontsize=13, fontweight='bold')
axes[2].legend(fontsize=9)

fig.suptitle('CHART 1 — Dataset Overview', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('INSIGHT: 87% ham / 13% spam (imbalanced). Spam messages tend to be longer.')


In [ ]:

# ── 3c: Chart 2 — Scatter plots: anomaly detection ────────────────────────
Q1 = df['char_count'].quantile(0.25)
Q3 = df['char_count'].quantile(0.75)
IQR = Q3 - Q1
outlier_mask = (df['char_count'] < Q1 - 1.5*IQR) | (df['char_count'] > Q3 + 1.5*IQR)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Scatter 1: word count vs char count — shows outliers
for label, grp in df.groupby('label'):
    axes[0].scatter(grp['word_count'], grp['char_count'],
                    c=COLORS[label], alpha=0.3, s=15, label=label.capitalize())
outliers = df[outlier_mask]
axes[0].scatter(outliers['word_count'], outliers['char_count'],
                facecolors='none', edgecolors='red', linewidths=1.5,
                s=90, zorder=5, label=f'Outlier (n={outlier_mask.sum()})')
axes[0].set_xlabel('Word count', fontsize=11)
axes[0].set_ylabel('Character count', fontsize=11)
axes[0].set_title('Word Count vs Char Count\n(red rings = statistical outliers beyond 1.5×IQR)',
                  fontsize=11, fontweight='bold')
axes[0].legend()

# Scatter 2: uppercase ratio vs exclamation marks — spam cluster
for label, grp in df.groupby('label'):
    axes[1].scatter(grp['upper_ratio'], grp['exclaim_count'],
                    c=COLORS[label], alpha=0.3, s=15, label=label.capitalize())
axes[1].set_xlabel('Uppercase ratio  (ALL CAPS / total chars)', fontsize=11)
axes[1].set_ylabel('Number of exclamation marks (!)', fontsize=11)
axes[1].set_title('Uppercase Ratio vs Exclamation Marks\n(spam clusters top-right = SHOUTING + hype!!!!)',
                  fontsize=11, fontweight='bold')
axes[1].legend()

fig.suptitle('CHART 2 — Anomaly & Spam Signal Detection', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
print('INSIGHT: Spam uses more capital letters AND more exclamation marks.')
print(f'         {outlier_mask.sum()} messages are outlier-length — mostly spam trying to cram in more content.')


In [ ]:

# ── 3d: Chart 3 — Box plots for all 6 numeric features ────────────────────
features = ['char_count','word_count','upper_ratio',
            'digit_count','exclaim_count','punct_count']
titles   = ['Character count','Word count','Uppercase ratio',
            'Digit count','Exclamation marks','Punctuation count']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, (feat, title) in enumerate(zip(features, titles)):
    sns.boxplot(data=df, x='label', y=feat, palette=COLORS, width=0.45,
                flierprops={'marker':'o','markerfacecolor':'grey','markersize':2,'alpha':0.3},
                ax=axes[i])
    for j, label in enumerate(['ham','spam']):
        med = df[df['label']==label][feat].median()
        axes[i].text(j, med + df[feat].max()*0.02, f'{med:.1f}',
                     ha='center', fontsize=9, fontweight='bold', color='black')
    axes[i].set_title(title, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('')

fig.suptitle('CHART 3 — Feature Distributions: Ham vs Spam  (medians labelled)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('INSIGHT: Every single feature separates spam from ham — the model has rich signals to learn from.')


In [ ]:

# ── 3e: Chart 4 — URL and money/prize language by class ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, feat, title in zip(
        axes,
        ['has_url', 'has_currency'],
        ['Contains a URL / web link', 'Contains money/prize words  (FREE, WIN, CASH, £…)']):
    pct = df.groupby('label')[feat].mean() * 100
    bars = ax.bar(pct.index, pct.values,
                  color=[COLORS[l] for l in pct.index],
                  edgecolor='white', width=0.4)
    for bar, val in zip(bars, pct.values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.8, f'{val:.1f}%',
                ha='center', fontweight='bold', fontsize=13)
    ax.set_ylabel('% of messages in that class')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylim(0, pct.max() * 1.35)

fig.suptitle('CHART 4 — Spam Red Flags: URLs and Money Language', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('INSIGHT: Spam is ~8× more likely to contain a URL and ~7× more likely to use prize/money words.')


In [ ]:

# ── 3f: Chart 5 — Top 15 words per class ──────────────────────────────────
STOPWORDS = {
    'i','a','the','to','and','is','in','it','of','you','my','me','we',
    'he','she','are','be','was','for','on','that','this','with','do',
    'have','your','but','not','at','or','from','an','as','so','if',
    'up','its','no','by','had','has','all','any','there','just','get',
    'now','will','can','dont','im','got','how','what','ok','yes','s',
    'u','r','2','4','ur','am','then','when','been','they','their',
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, label in zip(axes, ['ham', 'spam']):
    text  = df[df['label']==label]['text'].str.lower().str.cat(sep=' ')
    words = re.findall(r'[a-z]+', text)
    words = [w for w in words if w not in STOPWORDS and len(w) > 1]
    top   = Counter(words).most_common(15)
    w_list, counts = zip(*top)

    bars = ax.barh(range(len(w_list)), counts, color=COLORS[label], alpha=0.85, edgecolor='white')
    ax.set_yticks(range(len(w_list)))
    ax.set_yticklabels(w_list, fontsize=11)
    ax.invert_yaxis()
    ax.set_xlabel('Frequency')
    ax.set_title(f'Top 15 words — {label.upper()}', fontsize=13,
                 fontweight='bold', color=COLORS[label])
    for bar, count in zip(bars, counts):
        ax.text(bar.get_width() + max(counts)*0.01,
                bar.get_y() + bar.get_height()/2,
                str(count), va='center', fontsize=9)

fig.suptitle('CHART 5 — Most Frequent Words: Ham vs Spam', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('INSIGHT: Spam vocabulary → call, free, txt, claim, prize, reply.')
print('         Ham vocabulary  → home, love, going, come, time, good.')


In [ ]:

# ── 3g: Chart 6 — Correlation heatmap ─────────────────────────────────────
feat_cols = ['label_id','char_count','word_count','unique_words',
             'upper_ratio','digit_count','exclaim_count',
             'punct_count','has_url','has_currency']
corr = df[feat_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))   # lower triangle only
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 9})
ax.set_title('CHART 6 — Feature Correlation Matrix\n'
             '(label_id = 1 means spam  |  warm = positive correlation)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('INSIGHT: has_currency (+0.37), upper_ratio (+0.30), exclaim_count (+0.24) and has_url (+0.22)')
print('         are the strongest individual predictors of spam.')


In [ ]:

# ── 3h: Chart 7 — Outlier deep-dive: violin + who are the outliers? ────────
Q1 = df['char_count'].quantile(0.25)
Q3 = df['char_count'].quantile(0.75)
fence = Q3 + 1.5 * (Q3 - Q1)
outlier_df = df[df['char_count'] > fence]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Violin plot
sns.violinplot(data=df, x='label', y='char_count', palette=COLORS,
               inner='quartile', ax=axes[0])
axes[0].axhline(fence, color='red', linestyle='--', linewidth=1.5,
                label=f'Outlier fence  ({fence:.0f} chars)')
axes[0].set_title('Message Length — Violin Plot\n(red line = statistical outlier boundary)',
                  fontsize=11, fontweight='bold')
axes[0].set_ylabel('Character count')
axes[0].legend()

# % of each class that are outliers
outlier_pct = (outlier_df['label'].value_counts() / df['label'].value_counts() * 100).fillna(0)
bars = axes[1].bar(outlier_pct.index, outlier_pct.values,
                   color=[COLORS[l] for l in outlier_pct.index],
                   edgecolor='white', width=0.4)
for bar, val in zip(bars, outlier_pct.values):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.3, f'{val:.1f}%',
                 ha='center', fontweight='bold', fontsize=13)
axes[1].set_title('% of each class that exceed the outlier fence',
                  fontsize=11, fontweight='bold')
axes[1].set_ylabel('% of messages in class')
axes[1].set_ylim(0, outlier_pct.max() * 1.35)
axes[1].set_xlabel('')

fig.suptitle('CHART 7 — Outlier Analysis: Who sends very long messages?',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Outlier messages (>{fence:.0f} chars): {len(outlier_df)}')
print(outlier_df['label'].value_counts().to_string())
if len(outlier_df[outlier_df['label']=='spam']) > 0:
    sample = outlier_df[outlier_df['label']=='spam']['text'].iloc[0]
    print(f'\nSample outlier SPAM ({len(sample)} chars):\n  {sample[:140]}...')


In [ ]:

# ── 3i: Chart 8 — Pairplot of top 4 discriminating features ───────────────
import warnings
warnings.filterwarnings('ignore')

pair_df = df[['label','char_count','upper_ratio','exclaim_count','has_currency']].copy()
pair_df.columns = ['label','Char count','Uppercase ratio','Exclamations','Has money words']

g = sns.pairplot(pair_df, hue='label', palette=COLORS,
                 plot_kws={'alpha': 0.25, 's': 12},
                 diag_kind='kde', corner=True)
g.figure.suptitle('CHART 8 — Pairplot: How 4 Features Separate Ham from Spam',
                  fontsize=14, fontweight='bold', y=1.01)
plt.show()
print('INSIGHT: Any two of these four features already separate most spam from ham.')
print('         DistilBERT learns these AND subtler language patterns simultaneously.')


In [ ]:

# ── 3i: EDA Summary ────────────────────────────────────────────────────────
summary = {
    'Total messages':              len(df),
    'Ham (legitimate)':            int((df['label']=='ham').sum()),
    'Spam':                        int((df['label']=='spam').sum()),
    'Class imbalance ratio':       f"{(df['label']=='ham').sum() / (df['label']=='spam').sum():.1f} : 1  (ham:spam)",
    'Avg ham length (chars)':      f"{df[df['label']=='ham']['char_count'].mean():.0f}",
    'Avg spam length (chars)':     f"{df[df['label']=='spam']['char_count'].mean():.0f}",
    'Spam with URL (%)':           f"{df[df['label']=='spam']['has_url'].mean()*100:.1f}%",
    'Ham with URL (%)':            f"{df[df['label']=='ham']['has_url'].mean()*100:.1f}%",
    'Spam with money words (%)':   f"{df[df['label']=='spam']['has_currency'].mean()*100:.1f}%",
    'Ham with money words (%)':    f"{df[df['label']=='ham']['has_currency'].mean()*100:.1f}%",
    'Statistical outliers':        int(outlier_df.shape[0]),
    'Missing values':              int(df.isnull().sum().sum()),
}

print('=' * 56)
print('  EDA SUMMARY')
print('=' * 56)
for k, v in summary.items():
    print(f'  {k:<38} {v}')
print('=' * 56)
print('''
WHY THIS DATA IS PERFECT FOR TRANSFER LEARNING
  ✓ Small (5,574 rows) → must reuse pre-trained weights
  ✓ Clean binary labels, zero missing values
  ✓ Strong lexical signals (FREE, WIN, £, URLs)
  ✓ Structural signals (length, caps, punctuation)
  ✓ Real-world useful — every email server does this

NEXT STEP → Cell 4: Tokenise with DistilBERT and split 70/15/15
''')


## Cell 4 — Tokenise + 70/15/15 split

In [ ]:
import json, pandas as pd
from pathlib import Path
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast

PROCESSED_DIR = Path('data/processed')
CHECKPOINT = 'distilbert-base-uncased'
MAX_LENGTH = 128

df = pd.read_csv('data/raw/spam.csv')
if 'v1' in df.columns:
    df = df[['v1','v2']].rename(columns={'v1':'label','v2':'text'})
df = df[['label','text']].dropna()
df['label'] = df['label'].str.strip().str.lower()
df['labels'] = (df['label'] == 'spam').astype(int)

train_df, tmp_df = train_test_split(df, test_size=0.30, stratify=df['labels'], random_state=42)
val_df, test_df = train_test_split(tmp_df, test_size=0.50, stratify=tmp_df['labels'], random_state=42)
print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

raw_ds = DatasetDict({
    'train': Dataset.from_pandas(train_df[['text','labels']], preserve_index=False),
    'val':   Dataset.from_pandas(val_df[['text','labels']],   preserve_index=False),
    'test':  Dataset.from_pandas(test_df[['text','labels']],  preserve_index=False),
})

print(f'Loading tokenizer: {CHECKPOINT}...')
tokenizer = DistilBertTokenizerFast.from_pretrained(CHECKPOINT)

def tokenise(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=MAX_LENGTH)

tokenised_ds = raw_ds.map(tokenise, batched=True)
tokenised_ds.set_format('torch', columns=['input_ids','attention_mask','labels'])

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
tokenised_ds.save_to_disk(str(PROCESSED_DIR))
Path(PROCESSED_DIR / 'label_map.json').write_text(json.dumps({'0':'ham','1':'spam'}))
print(f'Tokenised dataset saved to {PROCESSED_DIR}')

## Cell 5 — Fine-tune DistilBERT (≈10 min on CPU, ≈90 sec on GPU)

In [ ]:
import numpy as np, mlflow
from pathlib import Path
from datasets import load_from_disk
from sklearn.metrics import accuracy_score, f1_score
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast, Trainer, TrainingArguments

CHECKPOINT = 'distilbert-base-uncased'
BEST_DIR   = Path('models/best')
BEST_DIR.mkdir(parents=True, exist_ok=True)

ds = load_from_disk('data/processed')
ds.set_format('torch', columns=['input_ids','attention_mask','labels'])

model     = DistilBertForSequenceClassification.from_pretrained(CHECKPOINT, num_labels=2)
tokenizer = DistilBertTokenizerFast.from_pretrained(CHECKPOINT)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds), 'f1': f1_score(labels, preds)}

args = TrainingArguments(
    output_dir='models/checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=50,
    report_to='none',
    seed=42,
)

trainer = Trainer(model=model, args=args,
                  train_dataset=ds['train'], eval_dataset=ds['val'],
                  tokenizer=tokenizer, compute_metrics=compute_metrics)

mlflow.set_experiment('sms-spam-distilbert')
with mlflow.start_run(run_name='full-fine-tune'):
    mlflow.log_params({'epochs':3,'lr':2e-5,'batch':32,'frozen':'none'})
    result = trainer.train()
    val_m  = trainer.evaluate(ds['val'])
    mlflow.log_metrics({'val_accuracy': val_m['eval_accuracy'], 'val_f1': val_m['eval_f1']})
    trainer.save_model(str(BEST_DIR))
    tokenizer.save_pretrained(str(BEST_DIR))

print(f"\nDone. Val accuracy: {val_m['eval_accuracy']:.4f}  F1: {val_m['eval_f1']:.4f}")
print(f'Model saved to {BEST_DIR}')

## Cell 6 — Evaluate on validation set + confusion matrix

In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from datasets import load_from_disk
from sklearn.metrics import classification_report, confusion_matrix
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast, Trainer, TrainingArguments

ds        = load_from_disk('data/processed')
ds.set_format('torch', columns=['input_ids','attention_mask','labels'])
model     = DistilBertForSequenceClassification.from_pretrained('models/best')
tokenizer = DistilBertTokenizerFast.from_pretrained('models/best')

trainer = Trainer(model=model, tokenizer=tokenizer,
                  args=TrainingArguments(output_dir='models/eval_tmp',
                                        per_device_eval_batch_size=64,
                                        report_to='none'))
preds  = trainer.predict(ds['val'])
y_pred = np.argmax(preds.predictions, axis=-1)
y_true = preds.label_ids

print(classification_report(y_true, y_pred, target_names=['ham','spam']))

fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=['ham','spam'], yticklabels=['ham','spam'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Validation')
plt.tight_layout(); plt.show()

## Cell 7 — Freeze 5 layers + re-tune (only 3.5% of params)

In [ ]:
import numpy as np, mlflow
from pathlib import Path
from datasets import load_from_disk
from sklearn.metrics import accuracy_score, f1_score
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast, Trainer, TrainingArguments

FROZEN_DIR = Path('models/frozen')
FROZEN_DIR.mkdir(parents=True, exist_ok=True)

ds        = load_from_disk('data/processed')
ds.set_format('torch', columns=['input_ids','attention_mask','labels'])
model     = DistilBertForSequenceClassification.from_pretrained('models/best')
tokenizer = DistilBertTokenizerFast.from_pretrained('models/best')

# Freeze embeddings + first 5 transformer layers
for param in model.distilbert.embeddings.parameters():
    param.requires_grad = False
for i in range(5):
    for param in model.distilbert.transformer.layer[i].parameters():
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.1f}%)')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds), 'f1': f1_score(labels, preds)}

args = TrainingArguments(
    output_dir='models/frozen_ckpt',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=50,
    report_to='none',
    seed=42,
)
trainer = Trainer(model=model, args=args,
                  train_dataset=ds['train'], eval_dataset=ds['val'],
                  tokenizer=tokenizer, compute_metrics=compute_metrics)

mlflow.set_experiment('sms-spam-distilbert')
with mlflow.start_run(run_name='frozen-5-layers'):
    mlflow.log_params({'epochs':3,'lr':3e-5,'batch':32,'frozen_layers':5})
    trainer.train()
    val_m = trainer.evaluate(ds['val'])
    mlflow.log_metrics({'val_accuracy': val_m['eval_accuracy'], 'val_f1': val_m['eval_f1']})
    trainer.save_model(str(FROZEN_DIR))
    tokenizer.save_pretrained(str(FROZEN_DIR))

print(f"\nFrozen model — Val accuracy: {val_m['eval_accuracy']:.4f}  F1: {val_m['eval_f1']:.4f}")

## Cell 8 — Final test evaluation (run this ONCE)

In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from datasets import load_from_disk
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast, Trainer, TrainingArguments

# Use frozen model if available, else best
model_path = 'models/frozen' if Path('models/frozen').exists() else 'models/best'
print(f'Loading model from: {model_path}')

ds        = load_from_disk('data/processed')
ds.set_format('torch', columns=['input_ids','attention_mask','labels'])
model     = DistilBertForSequenceClassification.from_pretrained(model_path)
tokenizer = DistilBertTokenizerFast.from_pretrained(model_path)

trainer = Trainer(model=model, tokenizer=tokenizer,
                  args=TrainingArguments(output_dir='models/test_tmp',
                                        per_device_eval_batch_size=64,
                                        report_to='none'))
preds  = trainer.predict(ds['test'])
y_pred = np.argmax(preds.predictions, axis=-1)
y_true = preds.label_ids

print('\n========== FINAL TEST RESULTS ==========')
print(classification_report(y_true, y_pred, target_names=['ham','spam']))
print(f'Accuracy : {accuracy_score(y_true, y_pred):.4f}')
print(f'F1 (spam): {f1_score(y_true, y_pred):.4f}')
print('=========================================')

fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Oranges',
            xticklabels=['ham','spam'], yticklabels=['ham','spam'], ax=ax)
ax.set_title('Confusion Matrix — Test Set (Final)')
plt.tight_layout(); plt.show()

## Cell 9 — Quick inference (test any message)

In [ ]:
import torch
from pathlib import Path
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

model_path = 'models/frozen' if Path('models/frozen').exists() else 'models/best'
model     = DistilBertForSequenceClassification.from_pretrained(model_path)
tokenizer = DistilBertTokenizerFast.from_pretrained(model_path)
model.eval()

def predict(text: str) -> dict:
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        logits = model(**enc).logits
    probs = torch.softmax(logits, dim=-1)[0]
    label = 'spam' if probs[1] > probs[0] else 'ham'
    return {'label': label, 'confidence': f'{probs.max().item():.2%}'}

# Try your own messages here!
test_messages = [
    'Congratulations! You won a FREE iPhone. Click now to claim!',
    'Hey, are we still on for dinner tonight?',
    'URGENT: Your account has been suspended. Verify now at http://scam.com',
    'Can you pick up some milk on your way home?',
]

for msg in test_messages:
    result = predict(msg)
    print(f"[{result['label'].upper():4s}] {result['confidence']}  →  {msg[:60]}")